THIẾT LẬP HÀM HUẤN LUYỆN

In [2]:
import numpy as np
from scipy.sparse import issparse
import matplotlib.pyplot as plt

In [ ]:
class SVMFromScratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = 0
        self.epoch_log = [] # Lưu số epoch đã chạy
        self.loss_log = [] # Lưu giá trị loss sau mỗi epoch để theo dõi quá trình hội tụ

    def fit(self, X, y):
        """
        Huấn luyện mô hình SVM sử dụng Stochastic Gradient Descent (SGD).
        X: sparse (scipy sparse matrix).
        y: nhãn lớp, được chuyển đổi thành -1 và 1.
        Thuật toán:
        1. Khởi tạo trọng số w và bias b.
        2. Với mỗi epoch:
           - Với mỗi mẫu x_i:
             - Tính decision = w.x_i + b.
             - Nếu y_i * decision >= 1: cập nhật w và b theo hướng giảm thiểu regularization.
             - Ngược lại: cập nhật w và b theo hướng giảm thiểu cả regularization và hinge loss.
        3. Tính loss sau mỗi epoch để theo dõi quá trình hội tụ.
        """
        n_samples, n_features = X.shape
        y_transformed = np.where(y <= 0, -1, 1)

        # Khởi tạo trọng số w và bias b 
        # self.w là vector có kích thước bằng số đặc trưng, khởi tạo bằng 0 --> là Dense Array gồm 10000 phần tử
        self.w = np.zeros(n_features)
        self.b = 0

        self.epoch_log = []
        self.loss_log  = []

        for epoch in range(self.n_iters):
            total_loss = 0

            for idx in range(n_samples):
                x_i = X[idx]

                """
                Tính decision value: w.x_i + b.
                    x_i.dot(self.w): lấy vecto dòng x_i (chứa điểm TF-IDF) nhân vô hướng với vecto trọng số self.w --> trả về ma trận 1x1 có dạng [[value]]
                    np.asarray() chuyển ma trận 1x1 thành mảng numpy, 
                    ravel() làm phẳng mảng thành vector 1D dạng [value],
                    [0] lấy giá trị đầu tiên.
                    + self.b: cộng thêm bias.
                """
                decision = np.asarray(x_i.dot(self.w)).ravel()[0] + self.b
                condition = y_transformed[idx] * decision >= 1

                if condition:
                    self.w -= self.learning_rate * (2 * self.lambda_param * self.w)
                else:
                    """
                    Vì ta sẽ lấy mảng đặc (self.w) nhân với một hàng của ma trận sparse (x_i), 
                    --> Ta cần chuyển hàng đó thành mảng đặc để thực hiện phép nhân vô hướng.
                    x_i.toarray() chuyển hàng sparse thành mảng 2D có một hàng,
                    ravel() làm phẳng thành vector 1D.
                    """
                    x_i_dense = x_i.toarray().ravel() if issparse(x_i) else x_i
                    self.w -= self.learning_rate * (
                        2 * self.lambda_param * self.w - x_i_dense * y_transformed[idx]
                    )
                    self.b += self.learning_rate * y_transformed[idx]
                    # Hinge loss của mẫu này
                    total_loss += max(0, 1 - y_transformed[idx] * decision)

            # Tính các chỉ số sau mỗi epoch
            total_loss = total_loss / n_samples + self.lambda_param * np.dot(self.w, self.w)

            # Log mỗi 10 epoch
            if (epoch + 1) % 1 == 0 or epoch == 0:
                print(f"{epoch+1:>4} | {total_loss: .4f}")
                self.epoch_log.append(epoch + 1)
                self.loss_log.append(total_loss)

        print("Huấn luyện thành công!")

    def predict(self, X):
        """
        Dự đoán nhãn cho dữ liệu đầu vào X.
        X: sparse (scipy sparse matrix).
        Thuật toán:
            1. Tính linear_output = X.dot(w) + b.
            - X.dot(w) hoạt động trực tiếp với sparse matrix, trả về một vector kết quả có kích thước bằng số mẫu, mỗi phần tử là kết quả của w.x_i.
            2. Sử dụng np.sign để xác định dấu của linear_output:
                - Nếu linear_output > 0, np.sign trả về 1 (dự đoán lớp dương).
                - Nếu linear_output <= 0, np.sign trả về 0 (dự đoán lớp âm).
            3. Trả về mảng dự đoán nhãn (0 hoặc 1) tương ứng với từng mẫu trong X.
                - Nếu dấu dương, trả về 1 --> Spam
                - Nếu dấu âm hoặc bằng 0, trả về 0. --> Not Spam
        """
        # X.dot(self.w) hoạt động trực tiếp với sparse matrix
        linear_output = np.asarray(X.dot(self.w)).ravel() + self.b
        return np.where(np.sign(linear_output) > 0, 1, 0)
    
    def plot_loss(self):
        plt.figure(figsize=(10, 5))
        plt.plot(self.epoch_log, self.loss_log,
                 color='steelblue', linewidth=2, marker='o', markersize=3)
        plt.title('Quá trình hội tụ của mô hình SVM', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

LẤY DỮ LIỆU TRAIN/TEST

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
# BƯỚC 1: ĐỌC DỮ LIỆU THÔ TỪ FILE CSV
print("Bắt đầu đọc dữ liệu từ các file CSV...")

train_df = pd.read_csv('../../DATA/DATA_PREPROCESSING/EMAIL/CSV/train_balance.csv')
test_df  = pd.read_csv('../../DATA/DATA_PREPROCESSING/EMAIL/CSV/test_balance.csv')

X_train_raw = train_df['text'].values
X_test_raw  = test_df['text'].values

# Chuyển cột 'spam' thành mảng numpy để dễ dàng xử lý sau này
y_train     = np.array(train_df['spam'].values) 
y_test      = np.array(test_df['spam'].values) 

print(f"Đã nạp dữ liệu thành công!")
print(f"Tập Train: {len(X_train_raw)} mẫu  |  Spam: {y_train.sum()}  |  Ham: {len(y_train) - y_train.sum()}")
print(f"Tập Test : {len(X_test_raw)} mẫu  |  Spam: {y_test.sum()}  |  Ham: {len(y_test) - y_test.sum()}")


# BƯỚC 2: TF-IDF VECTORIZATION (chuyển text → ma trận số)
print("\nĐang áp dụng TF-IDF Vectorization...")

tfidf = TfidfVectorizer(
    max_features=10000,      # Giữ 10.000 từ phổ biến nhất
    sublinear_tf=True,       # Dùng log(tf) — giảm ảnh hưởng từ lặp nhiều 
    token_pattern=r'\b\w{2,}\b', # Bỏ ký tự đơn, giữ từ ≥ 2 ký tự
    ngram_range=(1, 2),      # Unigram + Bigram để bắt ngữ cảnh
    min_df=3,                # Bỏ từ chỉ xuất hiện trong 3 email trở lên
    max_df=0.95              # Bỏ từ xuất hiện trong hơn 95% email (quá phổ biến, ít ý nghĩa)
)

# Quan trọng: chỉ fit trên TRAIN, transform TEST — tránh data leakage
X_train = tfidf.fit_transform(X_train_raw) # Chuyển văn bản thô thành ma trận TF-IDF sparse, dùng cho tập Train 
X_test  = tfidf.transform(X_test_raw) # Chuyển văn bản thô thành ma trận TF-IDF sparse, nhưng không fit lại để tránh data leakage, dùng cho tập Test

print(f"Kích thước tập Train sau TF-IDF: {X_train.shape}")
print(f"Kích thước tập Test  sau TF-IDF: {X_test.shape}")

# Kiểm tra mật độ của ma trận sparse để đảm bảo rằng nó thực sự sparse (nhiều phần tử bằng 0)
print(f"Mật độ sparse matrix: {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.2f}%") 

Bắt đầu đọc dữ liệu từ các file CSV...
Đã nạp dữ liệu thành công!
Tập Train: 6552 mẫu  |  Spam: 3249  |  Ham: 3303
Tập Test : 1639 mẫu  |  Spam: 813  |  Ham: 826

Đang áp dụng TF-IDF Vectorization...
Kích thước tập Train sau TF-IDF: (6552, 10000)
Kích thước tập Test  sau TF-IDF: (1639, 10000)
Mật độ sparse matrix: 0.84%


In [ ]:
# ==============================================================================
# CROSS-VALIDATION — TÌM BỘ THAM SỐ TỐI ƯU CHO SVM
# ==============================================================================
# Mục tiêu: Thay vì chọn learning_rate, lambda_param, n_iters ngẫu nhiên,
#            ta duyệt có hệ thống qua lưới tham số (Grid Search) kết hợp với
#            K-Fold Cross-Validation để chọn bộ tham số cho Accuracy cao nhất.
#
# Quy trình K-Fold Cross-Validation:
#   1. Chia tập Train thành K phần (fold) bằng nhau.
#   2. Với mỗi tổ hợp tham số:
#        - Lặp K lần: mỗi lần dùng 1 fold làm Validation, K-1 fold còn lại làm Train.
#        - Tính Accuracy trung bình trên K lần → CV Score của tổ hợp đó.
#   3. Chọn bộ tham số có CV Score cao nhất.
#   4. Huấn luyện lại trên toàn bộ tập Train với bộ tham số tốt nhất.
# ==============================================================================

from scipy.sparse import vstack as sparse_vstack

def cross_validate_svm(X, y, param_grid, n_folds=5):
    """
    Thực hiện K-Fold Cross-Validation để tìm bộ tham số tốt nhất cho SVMFromScratch.

    Tham số:
        X         : Ma trận đặc trưng sparse (scipy sparse matrix).
        y         : Mảng nhãn (0/1).
        param_grid: Dict chứa danh sách giá trị cần thử cho từng tham số.
                    Ví dụ: {
                        'learning_rate': [0.001, 0.01, 0.05],
                        'lambda_param' : [0.0001, 0.001, 0.01],
                        'n_iters'      : [5, 10, 20]
                    }
        n_folds   : Số fold K (mặc định 5).

    Trả về:
        best_params : Dict bộ tham số tốt nhất.
        best_score  : Accuracy trung bình cao nhất tìm được.
        results     : Danh sách kết quả của tất cả tổ hợp (để phân tích).
    """
    import itertools

    n_samples = X.shape[0]

    # Tạo tất cả tổ hợp tham số từ param_grid (Grid Search)
    keys   = list(param_grid.keys())
    values = list(param_grid.values())
    all_combinations = list(itertools.product(*values))
    total_combos = len(all_combinations)

    print(f"Tổng số tổ hợp tham số cần kiểm tra : {total_combos}")
    print(f"Số fold K                            : {n_folds}")
    print(f"Tổng số lần huấn luyện               : {total_combos * n_folds}")
    print("-" * 60)

    # Tạo chỉ số fold — chia đều n_samples thành n_folds phần
    # np.array_split chia mảng chỉ số thành n_folds mảng con (gần bằng nhau)
    indices    = np.arange(n_samples)
    fold_sizes = np.full(n_folds, n_samples // n_folds)
    fold_sizes[: n_samples % n_folds] += 1          # Phân bổ phần dư cho các fold đầu
    fold_indices = []
    current = 0
    for size in fold_sizes:
        fold_indices.append(indices[current : current + size])
        current += size

    results     = []
    best_score  = -1
    best_params = None

    for combo_idx, combo in enumerate(all_combinations):
        params = dict(zip(keys, combo))
        lr     = params['learning_rate']
        lam    = params['lambda_param']
        nit    = params['n_iters']

        fold_accuracies = []

        for fold_k in range(n_folds):
            # Chỉ số Validation = fold hiện tại; Train = các fold còn lại
            val_idx   = fold_indices[fold_k]
            train_idx = np.concatenate([fold_indices[j] for j in range(n_folds) if j != fold_k])

            X_cv_train = X[train_idx]
            y_cv_train = y[train_idx]
            X_cv_val   = X[val_idx]
            y_cv_val   = y[val_idx]

            # Huấn luyện mô hình — tắt print để không flood output
            import io, sys as _sys
            _old_stdout = _sys.stdout
            _sys.stdout = io.StringIO()
            try:
                model = SVMFromScratch(learning_rate=lr, lambda_param=lam, n_iters=nit)
                model.fit(X_cv_train, y_cv_train)
            finally:
                _sys.stdout = _old_stdout

            # Đánh giá trên fold Validation
            y_pred_val = model.predict(X_cv_val)
            acc = np.mean(y_pred_val == y_cv_val)
            fold_accuracies.append(acc)

        cv_score = np.mean(fold_accuracies)
        cv_std   = np.std(fold_accuracies)

        results.append({
            'learning_rate': lr,
            'lambda_param' : lam,
            'n_iters'      : nit,
            'cv_accuracy'  : cv_score,
            'cv_std'       : cv_std
        })

        print(f"[{combo_idx+1:>3}/{total_combos}] lr={lr}  λ={lam}  iters={nit:>3}"
              f"  →  CV Accuracy: {cv_score*100:.2f}% (±{cv_std*100:.2f}%)")

        if cv_score > best_score:
            best_score  = cv_score
            best_params = params.copy()

    print("-" * 60)
    print(f"Bộ tham số tốt nhất:")
    print(f"   learning_rate = {best_params['learning_rate']}")
    print(f"   lambda_param  = {best_params['lambda_param']}")
    print(f"   n_iters       = {best_params['n_iters']}")
    print(f"   CV Accuracy   = {best_score*100:.2f}%")

    return best_params, best_score, results


# ------------------------------------------------------------------------------
# ĐỊNH NGHĨA LƯỚI THAM SỐ CẦN TÌM KIẾM
# Tăng/giảm số giá trị trong mỗi danh sách tùy theo thời gian bạn có:
#   - Ít giá trị → nhanh hơn, phạm vi hẹp hơn.
#   - Nhiều giá trị → lâu hơn, tìm được tham số chính xác hơn.
# ------------------------------------------------------------------------------
param_grid = {
    'learning_rate': [0.001, 0.01, 0.05],   # Tốc độ học
    'lambda_param' : [0.0001, 0.0005, 0.001], # Hệ số regularization (L2)
    'n_iters'      : [5, 10, 20]             # Số epoch huấn luyện
}

print("=" * 60)
print("BẮT ĐẦU CROSS-VALIDATION TÌM THAM SỐ TỐI ƯU")
print("=" * 60)

best_params, best_cv_score, cv_results = cross_validate_svm(
    X_train, y_train,
    param_grid=param_grid,
    n_folds=5
)

BẮT ĐẦU CROSS-VALIDATION TÌM THAM SỐ TỐI ƯU
Tổng số tổ hợp tham số cần kiểm tra : 27
Số fold K                            : 5
Tổng số lần huấn luyện               : 135
------------------------------------------------------------
[  1/27] lr=0.001  λ=0.0001  iters=  5  →  Accuracy: 73.34% (±13.15%)  |  Recall: 46.69% (±25.70%)  |  FPR: 0.00% (±0.00%)
[  2/27] lr=0.001  λ=0.0001  iters= 10  →  Accuracy: 96.67% (±1.06%)  |  Recall: 99.81% (±0.23%)  |  FPR: 6.38% (±1.95%)
[  3/27] lr=0.001  λ=0.0001  iters= 20  →  Accuracy: 97.56% (±0.60%)  |  Recall: 99.78% (±0.23%)  |  FPR: 4.61% (±1.07%)
[  4/27] lr=0.001  λ=0.0005  iters=  5  →  Accuracy: 73.12% (±13.29%)  |  Recall: 46.27% (±25.97%)  |  FPR: 0.00% (±0.00%)
[  5/27] lr=0.001  λ=0.0005  iters= 10  →  Accuracy: 96.76% (±1.11%)  |  Recall: 99.81% (±0.23%)  |  FPR: 6.20% (±2.06%)
[  6/27] lr=0.001  λ=0.0005  iters= 20  →  Accuracy: 97.44% (±0.66%)  |  Recall: 99.78% (±0.23%)  |  FPR: 4.85% (±1.17%)
[  7/27] lr=0.001  λ=0.001  iters=  5  →

GIAI ĐOẠN HUẤN LUYỆN

In [13]:
# HUẤN LUYỆN MÔ HÌNH SVM
print("\nĐang tiến hành huấn luyện mô hình SVM từ gốc...")
svm = SVMFromScratch(learning_rate=0.05, lambda_param=0.0001, n_iters=10)
svm.fit(X_train, y_train)


Đang tiến hành huấn luyện mô hình SVM từ gốc...
   1 |  0.3270
   2 |  0.1209
   3 |  0.0972
   4 |  0.0862
   5 |  0.0803
   6 |  0.0764
   7 |  0.0733
   8 |  0.0712
   9 |  0.0696
  10 |  0.0683
Huấn luyện thành công!


THIẾT LẬP HÀM TEST BẰNG CÁC THƯỚC ĐO

In [9]:
def evaluate_model_comprehensive(y_true, y_pred):
    """
    Hàm tính toán toàn bộ các chỉ số hiệu năng từ gốc.
    Quy ước nhãn theo code trước: 1 là Spam (Positive), 0 là không Spam (Negative).
    """
    # Ép kiểu dữ liệu về dạng mảng phẳng Numpy để tính toán toán học
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # 1. Tính toán các thành phần của Ma trận nhầm lẫn (Confusion Matrix)
    TP = np.sum((y_true == 1) & (y_pred == 1)) # Thực tế là Spam, đoán đúng là Spam
    FP = np.sum((y_true == 0) & (y_pred == 1)) # Thực tế là không Spam, đoán sai thành Spam (Báo động giả!)
    TN = np.sum((y_true == 0) & (y_pred == 0)) # Thực tế là không Spam, đoán đúng là không Spam
    FN = np.sum((y_true == 1) & (y_pred == 0)) # Thực tế là Spam, đoán sai thành không Spam (Lọt lưới!)

    print("--- MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ---")
    print(f"True Positive (Bắt trúng Spam) : {TP}")
    print(f"False Positive (Chặn nhầm không Spam): {FP}")
    print(f"True Negative (Giữ đúng không Spam)  : {TN}")
    print(f"False Negative (Lọt lưới Spam) : {FN}\n")

    # 2. Tính toán các chỉ số (Thêm điều kiện tránh lỗi chia cho 0)
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0 # Tỷ lệ dự đoán đúng trên tổng số mẫu
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0 # Tỷ lệ dự đoán đúng trên tổng số mẫu được dự đoán là Spam
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0 # Tỷ lệ dự đoán đúng trên tổng số mẫu thực sự là Spam
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0 # Cân bằng giữa Precision và Recall, đặc biệt hữu ích khi dữ liệu bị lệch (imbalanced)
    
    # Chỉ số bổ sung nâng cao
    fpr = FP / (TN + FP) if (TN + FP) > 0 else 0 # Tỷ lệ chặn nhầm không Spam (False Positive Rate)

    # 3. Xuất kết quả trực quan
    print("--- CÁC CHỈ SỐ ĐÁNH GIÁ HIỆU NĂNG ---")
    print(f"1. Accuracy (Độ chính xác tổng thể)      : {accuracy * 100:.2f}%")
    print(f"2. Precision (Độ chuẩn xác bắt Spam)     : {precision * 100:.2f}%")
    print(f"3. Recall (Độ nhạy / Tỷ lệ bắt sạch Spam): {recall * 100:.2f}%")
    print(f"4. F1-Score (Cân bằng giữa Prec & Rec)   : {f1_score * 100:.2f}%")
    print(f"5. False Positive Rate (Tỷ lệ chặn nhầm) : {fpr * 100:.2f}%  <-- Càng gần 0% càng tốt")

    return {
        "accuracy": accuracy, "precision": precision, "recall": recall, 
        "f1_score": f1_score, "fpr": fpr
    }

BẮT ĐẦU TEST

In [15]:
y_pred = svm.predict(X_test)
metrics = evaluate_model_comprehensive(y_test, y_pred)

--- MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ---
True Positive (Bắt trúng Spam) : 811
False Positive (Chặn nhầm không Spam): 5
True Negative (Giữ đúng không Spam)  : 821
False Negative (Lọt lưới Spam) : 2

--- CÁC CHỈ SỐ ĐÁNH GIÁ HIỆU NĂNG ---
1. Accuracy (Độ chính xác tổng thể)      : 99.57%
2. Precision (Độ chuẩn xác bắt Spam)     : 99.39%
3. Recall (Độ nhạy / Tỷ lệ bắt sạch Spam): 99.75%
4. F1-Score (Cân bằng giữa Prec & Rec)   : 99.57%
5. False Positive Rate (Tỷ lệ chặn nhầm) : 0.61%  <-- Càng gần 0% càng tốt
